In [120]:
import pandas as pd 
import requests 
from io import StringIO

# %conda install lxml
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
url = 'https://en.wikipedia.org/wiki/List_of_footballers_with_500_or_more_goals'
res = requests.get(url, headers=headers).text
res = StringIO(res)

df = pd.read_html(res)
df

[    0                                                  1
 0 NaN  This section needs additional citations for ve...,
    Rank              Player    Club                  Country and other  \
    Rank              Player  League  Cup Continental Country and other   
 0     1   Cristiano Ronaldo  590[a]   57         172               143   
 1     2       Lionel Messi*  553[b]   71         157               115   
 2     3               Pelé*  604[c]   49          26                83   
 3     4             Romário  545[d]   93          54                64   
 4     5       Ferenc Puskás  516[e]   69          56                84   
 5     6        Josef Bican*  515[f]  137          38                32   
 6     7  Robert Lewandowski  423[g]   62         117                88   
 7     8        Jimmy Jones*  330[h]  286          14                 9   
 8     9        Gerd Müller*  405[i]   92          69                68   
 9    10       Joe Bambrick*  347[j]  253           5     

In [121]:
df = df[3]


In [122]:
df.to_csv('./data/top500goals.csv', index = False)

In [123]:
df = pd.read_csv('./data/top500goals.csv')
df.head()

,Rank,Player,Goals,Matches,Ratio,Career span
0,1,Erwin Helmchen,989+,582,1.70,1924–1951
1,2,Cristiano Ronaldo,975,1337,0.73,2002–present
2,3,Josef Bican,950+,624,1.52,1930–1957
3,4,Ronnie Rooke,934+,1030,0.91,1929–1961
4,5,Lionel Messi,925,1194,0.77,2003–present


In [124]:
import random, math

df['Career span'] = df['Career span'].str.split('–').str[0] # splits the string into a list of words

In [125]:
df['Career span'].isna().sum()

np.int64(0)

In [126]:
#df.loc[~df['Goals'].str.isnumeric()] tilde ~ means "not" so this is saying "locate the rows where the 'Goals' column is not numeric"
df['Goals'] = [item.replace('+', '') for item in df['Goals']] #item.replace first item is what is being replaced, second item is what it is being replaced with. In this case we are replacing the '+' with an empty string '' so that we can convert the column to numeric values
df['Goals']


0     989
1     975
2     950
3     934
4     925
     ... 
78    504
79    504
80    503
81    502
82    500
Name: Goals, Length: 83, dtype: object

In [127]:
df ['Matches'] = df['Matches'].str.replace('+', '').astype(int)
df.describe()

,Rank,Matches,Ratio
count,83.000000,83.000000,83.000000
mean,42.000000,724.915663,0.886024
std,24.103942,193.411996,0.267563
min,1.000000,324.000000,0.530000
25%,21.500000,590.500000,0.695000
50%,42.000000,711.000000,0.860000
75%,62.500000,836.500000,0.960000
max,83.000000,1337.000000,1.700000


In [128]:
df.loc[~df["Career span"].str.isnumeric()]

,Rank,Player,Goals,Matches,Ratio,Career span


In [129]:
df['Goals'] = df['Goals'].astype(int)
df['Career span'] = df['Career span'].astype(int)
df.head()

,Rank,Player,Goals,Matches,Ratio,Career span
0,1,Erwin Helmchen,989,582,1.70,1924
1,2,Cristiano Ronaldo,975,1337,0.73,2002
2,3,Josef Bican,950,624,1.52,1930
3,4,Ronnie Rooke,934,1030,0.91,1929
4,5,Lionel Messi,925,1194,0.77,2003


In [130]:
from datetime import datetime, timedelta
rows = []
columns = ['name', 'date', 'goals', 'assists']

for row in df.itertuples():
    name = row.Player
    start_date = f'{row._6}-08-01'
    start_date = datetime.strptime(start_date, '%Y-%m-%d')
    goal_ratio = row.Ratio
    goal_sd = random.random()
    goal_sd = goal_sd if goal_sd <0.7 else 0.7
    assist_sd = 1 - goal_sd
    assist_ratio = 1 - goal_ratio
    
    for i in range(row.Matches):
        name = name 
        match_date = timedelta(days = 3 * i) + start_date
        goals = random.normalvariate(goal_ratio, goal_sd)
        assists = random.normalvariate(assist_ratio, assist_sd)
        goals = round(goals) if goals > 0 else 0
        assists = round(assists) if assists > 0 else 0
        row = [name, match_date, goals, assists]
        rows.append(row)
games_df = pd.DataFrame(data = rows, columns = columns)
games_df.head()
        

,name,date,goals,assists
0,Erwin Helmchen,1924-08-01,2,0
1,Erwin Helmchen,1924-08-04,2,0
2,Erwin Helmchen,1924-08-07,2,0
3,Erwin Helmchen,1924-08-10,2,0
4,Erwin Helmchen,1924-08-13,2,0


In [131]:
games_df.shape

(60168, 4)

In [132]:
games_df.sample(10)

,name,date,goals,assists
42949,Hughie Ferguson,1915-10-25,1,0
49731,Giorgio Chinaglia,1963-10-31,1,0
12308,Sammy Hughes,1943-02-18,1,0
2349,Josef Bican,1934-02-11,0,0
14132,Tom Waring,1925-03-20,1,0
27841,Glenn Ferguson,1991-03-19,1,0
4475,Lionel Messi,2010-12-28,1,0
45896,Roberto Dinamite,1975-02-08,0,1
33474,José Torres,1959-01-28,1,0
13574,Ernst Wilimowski,1932-08-10,1,0


In [133]:
fake_df = games_df.groupby('name').agg(
    matches = ('name', 'count'),
    goals = ('goals', 'sum'),
    assists = ('assists', 'sum'),
    g_ratio = ('goals', 'mean'),
    a_ratio = ('assists', 'mean')
).reset_index().sort_values(by = 'goals', ascending = False).reset_index(drop = True)
# fake_df['goal_ratio'] = fake_df['goals'] / fake_df['matches']
fake_df.head(10)

,name,matches,goals,assists,g_ratio,a_ratio
0,Lionel Messi,1194,1194,577,1.000000,0.483250
1,Erwin Helmchen,582,1164,74,2.000000,0.127148
2,Cristiano Ronaldo,1337,1030,319,0.770381,0.238594
3,Ronnie Rooke,1030,955,95,0.927184,0.092233
4,Josef Bican,624,925,1,1.482372,0.001603
5,Jimmy Jones,760,856,102,1.126316,0.134211
6,Abe Lenstra,850,831,259,0.977647,0.304706
7,Romário,1003,807,264,0.804586,0.263210
8,Ferenc Deák,515,785,2,1.524272,0.003883
9,Robert Lewandowski,1079,781,301,0.723818,0.278962


In [135]:
for i in range(10):
    print(f'{i} df: {df.loc[i,["Player", "Goals"]]} | {i} fake: {fake_df.loc[i,["name", "goals"]]}')   

0 df: Player    Erwin Helmchen
Goals                989
Name: 0, dtype: object | 0 fake: name     Lionel Messi
goals            1194
Name: 0, dtype: object
1 df: Player    Cristiano Ronaldo
Goals                   975
Name: 1, dtype: object | 1 fake: name     Erwin Helmchen
goals              1164
Name: 1, dtype: object
2 df: Player    Josef Bican
Goals             950
Name: 2, dtype: object | 2 fake: name     Cristiano Ronaldo
goals                 1030
Name: 2, dtype: object
3 df: Player    Ronnie Rooke
Goals              934
Name: 3, dtype: object | 3 fake: name     Ronnie Rooke
goals             955
Name: 3, dtype: object
4 df: Player    Lionel Messi
Goals              925
Name: 4, dtype: object | 4 fake: name     Josef Bican
goals            925
Name: 4, dtype: object
5 df: Player    Jimmy Jones
Goals             840
Name: 5, dtype: object | 5 fake: name     Jimmy Jones
goals            856
Name: 5, dtype: object
6 df: Player    Ferenc Puskás
Goals               802
Name: 6, dtype

In [ ]:
|